# Building Your First AI Agent with LangChain: From Chatbot to Agent

In this tutorial, we'll explore the evolution from simple chatbots to powerful AI agents by building a pet gift finder. You'll understand the key differences and learn when to use each approach.

## Chatbots vs AI Agents: What's the Difference?

**Chatbots** are conversational interfaces that respond to user input based on their training data. They're great for:
- Answering questions from existing knowledge
- Having conversations
- Providing explanations and advice

**AI Agents** go beyond conversation - they can take actions in the real world using tools. They can:
- Search the web for current information
- Make API calls to external services
- Perform calculations and data analysis
- Execute code and interact with databases

## What You'll Learn
- Start with a simple model (chatbot approach)
- Identify limitations of knowledge-only responses
- Build custom tools for real-world capabilities
- Create a full AI agent with LangChain
- Add multimodal capabilities (text + images)
- Deploy to LangSmith for interactive use

## Prerequisites
- Basic Python knowledge
- OpenAI API key
- Tavily API key (for web search)
- LangSmith API key (optional, for deployment)

Let's start simple and build up!

## Step 1: Environment Setup

First, we'll load our environment variables and optionally enable LangSmith tracing.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# Optional: Enable LangSmith tracing for debugging and monitoring
# Uncomment these lines if you have LangSmith set up
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "pet-gift-finder-tutorial"

print("Environment loaded! LangSmith tracing available if configured.")

Environment loaded! LangSmith tracing available if configured.


## Step 2: Starting Simple - The Chatbot Approach

Let's begin with a basic language model to understand what chatbots can and cannot do.

In [2]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

# Initialize a basic chat model
model = init_chat_model(model="gpt-4o-mini")

# Create a system message to define the chatbot's role
system_message = SystemMessage(content="""
You are a helpful pet gift advisor. Help users find Christmas gifts for their pets 
based on your knowledge of pet products and care. Provide specific product suggestions 
when possible.
""")

print("Basic Pet Gift Chatbot initialized!")

Basic Pet Gift Chatbot initialized!


### Testing the Basic Chatbot

In [3]:
# Test the basic chatbot
user_question = HumanMessage(content="I have a playful orange tabby cat. What Christmas gifts would be perfect for him?")

response = model.invoke([system_message, user_question])
print("Chatbot Response:")
print(response.content)

Chatbot Response:
Oh, a playful orange tabby? Sounds like a lot of fun to shop for! Here are gift ideas that suit an energetic, curious cat and help channel all that play into enrichment and a happy Christmas.

Top picks by category

1) Interactive chase toys (great for a cat who loves to pounce)
- Go Cat Da Bird Wand Toy: classic feather wand on a string. Perfect for a tense, high-energy chase game. Easy to replace the bird lure as it wears.
- Cat Dancer 101 Cat Toy: simple, durable wand with a fabric/dance lure. Great for solo play if you don’t always have a human to play with.
- Kong Kickeroo Cat Toy: plush mouse/ball design with catnip; ideal for grabbing, kicking, and wrestling. Durable enough for wilder play sessions.

2) Brainy puzzle and treat toys (great for mental stimulation and curiosity)
- Catit Senses 2.0 Digger: a treat-dispensing toy that hides kibble and rewards pawing. Keeps a smart cat occupied and moving.
- Trixie Cat Activity Flip Board: small puzzle board with fli

### The Limitations Become Clear

Let's ask for something that requires current information:

In [4]:
# Ask for current pricing and availability
current_info_question = HumanMessage(content="""
What are the current prices for interactive cat toys on Amazon? 
Which ones are in stock right now and have good reviews?
""")

response = model.invoke([system_message, current_info_question])
print("Chatbot Response to Current Info Request:")
print(response.content)
print("\n" + "="*50)
print("LIMITATION: The chatbot can't access real-time information!")
print("It can only work with its training data, which has a cutoff date.")

Chatbot Response to Current Info Request:
I can’t pull live Amazon prices, stock status, or current reviews from here. But I can help you quickly find good options and point you to what to look for. I’ll also share a short list of solid interactive cat toys that typically get strong reviews, so you can search for them and compare easily.

How to check current prices, stock, and reviews on Amazon (quick guide)
- Search: “interactive cat toy” or narrow by type (e.g., “cat puzzle feeder,” “automatic laser toy,” “cat wand,” “motorized cat toy”).
- Use filters: In Stock, Prime, price range, and Customer Review stars (4.0+ is a good starting point).
- Sort by: Avg. Customer Review (high-to-low) or Best Seller.
- Check stock status on the product page (In Stock / Out of Stock, provide an estimated delivery date if available).
- Read a handful of top reviews (look for comments about durability, noise, pet engagement, and safety).
- Check Q&A and “What’s in the box” for included pieces and age/

## Step 3: Why We Need Agents

The chatbot approach has several limitations:

1. **No real-time information** - Can't check current prices, availability, or reviews
2. **No verification** - Can't confirm if products actually exist
3. **Static knowledge** - Information becomes outdated
4. **No actions** - Can't actually help you buy anything

**This is where AI agents shine!** Agents can use tools to:
- Search the web for current information
- Check real prices and availability
- Find the latest products and reviews
- Even help with purchasing (with the right integrations)

Let's build an agent that can actually help!

## Step 4: Creating Tools - The Agent's Superpowers

Now we'll give our AI the ability to take actions in the real world. Tools are Python functions that agents can call to perform specific tasks.

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# Initialize the Tavily client for web searching
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for current information about pet gifts, products, and shopping"""
    return tavily_client.search(query)

@tool
def pet_gift_search(pet_type: str, pet_characteristics: str, budget: str = "moderate", location: str = "US") -> Dict[str, Any]:
    """Search for specific pet gifts based on type, characteristics, budget, and location"""
    search_query = f"Christmas gifts for {pet_type} {pet_characteristics} {budget} budget 2024 where to buy {location}"
    return tavily_client.search(search_query)

@tool
def local_store_search(product_name: str, location: str) -> Dict[str, Any]:
    """Find local pet stores and retailers in a specific location that might carry a product"""
    search_query = f"pet stores near {location} {product_name} in stock local retailers"
    return tavily_client.search(search_query)

print("Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.")

Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.


### Testing Our Tools

Let's test one of our tools directly to see how it works:

In [6]:
# Test our tool directly
test_result = web_search.invoke({"query": "best interactive cat toys 2024 Amazon"})
print("Direct tool result (first result):")
print(f"Title: {test_result['results'][0]['title']}")
print(f"URL: {test_result['results'][0]['url']}")
print(f"Content: {test_result['results'][0]['content'][:200]}...")
print("\nThis is real, current information from the web!")

Direct tool result (first result):
Title: What Are the Best Cat Toys for 2024
URL: https://cats-mode.com/blogs/cat-thing-blog/what-are-the-best-cat-toys-for-2024?srsltid=AfmBOoqARt0qfIQRNMgoZZdRCqwjO9MAECe62Edmk-6TEsFxPIE5V7q5
Content: What Are the Best Cat Toys for 2024 · YVE LIFE Automatic Cat Laser Toy - $25 at Amazon · Potaroma Electric Flopping Fish – $12.98-$13.99 at Amazon....

This is real, current information from the web!


## Step 5: Designing the Agent's System Prompt

The system prompt is crucial for agents - it needs to explain not just the role, but also how to use tools effectively.

In [7]:
system_prompt = """
You are a helpful pet gift advisor with access to real-time web search capabilities.

Your role:
- Help pet owners find appropriate Christmas gifts based on their pet's characteristics
- Use your web search tools to find current products, prices, and availability
- Consider pet safety, size, age, and personality when making recommendations
- Provide specific product suggestions with purchasing information
- Offer options across different budget ranges
- Help users find local stores and retailers in their area

When to use tools:
- Use web_search for general queries about pet products, reviews, or shopping
- Use pet_gift_search when you have specific pet characteristics, budget, and location info
- Use local_store_search to find nearby pet stores that might carry specific products
- Always ask for the user's location if they want local shopping options
- Always search for current information rather than relying on outdated knowledge

Location examples:
- For US users: Search Amazon, Petco, PetSmart, Chewy
- For Philippines users: Search Shopee, Lazada, local pet stores in Manila/Cebu/Davao
- For other countries: Adapt to local e-commerce and pet store chains

When analyzing pets from photos:
- Identify breed characteristics that might influence gift choices
- Estimate size and age if possible
- Note any visible personality traits or energy levels

Always prioritize pet safety and provide real, purchasable products with current pricing and local availability.
"""

print("System prompt created with location-aware instructions!")

System prompt created with location-aware instructions!


## Step 6: Creating the AI Agent

Now we'll combine everything into a powerful AI agent:

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create our AI agent with tools
agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search, pet_gift_search, local_store_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # Enables conversation memory
)

print("AI Agent created! It now has location-aware search capabilities.")

AI Agent created! It now has location-aware search capabilities.


## Step 7: Testing the Agent

Let's test our agent with the same questions we asked the basic chatbot:

In [9]:
from langchain.messages import HumanMessage

# Configuration for conversation memory
config = {"configurable": {"thread_id": "pet_gift_session_1"}}

# Ask the same question we asked the basic chatbot
response = agent.invoke(
    {"messages": [HumanMessage(content="I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?")]},
    config
)

print("AI Agent Response (with real-time web search):")
print(response['messages'][-1].content)

AI Agent Response (with real-time web search):
That sounds like a fun, energetic guy to shop for! Here are tailored Christmas gift ideas for a playful adult orange tabby who loves to chase things and knock things off tables. I’ve split them by budget and added current online options in the US.

Top picks for a chase-happy adult cat

- Da Bird Cat Teaser Wand (Go Cat Da Bird)
  - Why it fits: classic feather wand that turns lurks into high-speed chases. Great for an active, tipped-tabby like yours.
  - Price range: roughly $10–$13
  - Where to buy (examples):
    - The Cat Connection: $10.25
    - The Cat House Inc: around $12.00
    - Natural Pet Center: around $9.99
  - Safety tip: supervise play and replace feathers if worn; store away when not in use to avoid accidental entanglement.

- SmartyKat Hot Pursuit Electronic Concealed Motion Cat Toy
  - Why it fits: hidden motion and lights spark sprint-and-pounce play, perfect for a cat who loves to chase.
  - Price range: about $12–$30 

### Testing Current Information - The Agent's Superpower

In [10]:
# Ask for current pricing and availability - the question that stumped the chatbot
current_info_response = agent.invoke(
    {"messages": [HumanMessage(content="What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?")]},
    config
)

print("AI Agent Response to Current Info Request:")
print(current_info_response['messages'][-1].content)
print("\n" + "="*50)
print("SUCCESS: The agent can access real-time information!")
print("It searched the web and found current prices and availability.")

AI Agent Response to Current Info Request:
Here’s a quick snapshot of interactive cat toys you’ll often find on Amazon, with current price cues and notes on stock and reviews. Prices on Amazon change frequently, so I’ve also included links so you can see the exact price and stock status right now.

1) SmartyKat Hot Pursuit Electronic Concealed Motion Cat Toy
- What it is: a battery-powered toy with hidden motion and lights to trigger chasing instinct.
- Current price snapshot: typically around $12–$20 when on sale; MSRP listed by the maker is $29.99. NBC News has listed a sale figure around $14.28 in their roundup.
- Stock: usually listed on Amazon and commonly in stock, but stock can fluctuate.
- Reviews: widely reviewed; generally well-received for keeping energetic cats engaged.
- Amazon links to check: 
  - Product listing hub: https://www.amazon.com/smartykat/s?k=smartykat
  - Price/history reference: https://camelcamelcamel.com/SmartyKat-Hot-Pursuit-Concealed-Motion/product/B00EZ

## Step 8: Location-Aware Shopping

Let's test the location-aware capabilities with different regions:

In [11]:
# Example for Philippines users
philippines_response = agent.invoke(
    {"messages": [HumanMessage(content="I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.")]},
    config
)

print("Philippines Shopping Response:")
print(philippines_response['messages'][-1].content)

Philippines Shopping Response:
Nice—Filipino e-commerce has several solid options in your budget for a small breed like a Shih Tzu. Here are current picks you can find locally (Lazada and Shopee) with price ranges to help you stay within 1000–2000 PHP. I’ve included direct links so you can check stock and delivery times near Manila.

Budget-friendly and reliable picks (approx. 1000–2000 PHP)

- KONG Wobbler Dog Toy (interactive treat-dispensing)
  - Why it’s good: slows down feeding, mentally engaging, great for small dogs.
  - Price range on Lazada PH: around ₱1,470–₱1,771 depending on seller.
  - Links to check:
    - https://www.lazada.com.ph/products/kong-wobbler-dog-toy-i1389196304.html (about ₱1,470)
    - https://www.lazada.com.ph/products/kong-wobbler-i4250321514.html (about ₱1,470–₱1,771)
  - Notes: sizes vary—choose a small/medium version for a Shih Tzu. Check the size on the product page.

- Gigwi Wild Hunter Series with Treats Dog Toy
  - Why it’s good: interactive treat-di

In [12]:
# Example for finding local stores
local_store_response = agent.invoke(
    {"messages": [HumanMessage(content="I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?")]},
    config
)

print("Local Store Finder Response:")
print(local_store_response['messages'][-1].content)

Local Store Finder Response:
Absolutely. Here are Austin-area stores that typically carry puzzle feeders and interactive cat toys, plus notes on what you’re likely to find and how to check stock.

Stores to check near Austin, TX

- PetSmart (multiple locations in the Austin area)
  - Likely in-store options: Nina Ottosson cat puzzles such as Buggin’ Out (puzzle feeder) and other Nina Ottosson cat puzzles.
  - Example product page (in-store stock varies): Catstages Nina Ottosson Buggin' Out Puzzle Cat Toy
  - Price indicator: around $20.19 on the PetSmart site for the Buggin’ Out model
  - Austin location notes: curbside and in-store pickup available at many stores
  - Check here for the product: https://www.petsmart.com/cat/toys/interactive-and-electronic/catstages-nina-ottosson-buggin-out-puzzle-cat-toy-88731.html

- Petco (Austin area)
  - Likely options: Nina Ottosson puzzle feeders and other cat puzzles in the interactive toys category
  - Store locator: https://stores.petco.com/tx

## Step 9: Streaming Responses for Better UX

For longer responses, streaming provides a better user experience:

In [13]:
# Streaming response example
def stream_agent_response(message, config):
    """Stream the agent's response for better user experience"""
    print("Agent is thinking and searching...\n")
    
    # Stream the response
    for chunk in agent.stream({"messages": [HumanMessage(content=message)]}, config):
        # Print each chunk as it arrives
        if "messages" in chunk and chunk["messages"]:
            last_message = chunk["messages"][-1]
            if hasattr(last_message, 'content') and last_message.content:
                print(last_message.content, end="", flush=True)
    print("\n\nResponse complete!")

# Test streaming with a complex query
stream_agent_response(
    "I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility?",
    config
)

Agent is thinking and searching...





Response complete!


## Step 10: Adding Multimodal Capabilities

Let's add image analysis so users can upload photos of their pets:

In [14]:
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(
    accept='.png,.jpg,.jpeg', 
    multiple=False,
    description='Upload Pet Photo'
)
display(uploader)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload Pet Photo')

In [15]:
def process_uploaded_image():
    """Convert uploaded image to base64 format for the AI model"""
    if not uploader.value:
        return None, None
    
    # Get the uploaded file
    uploaded_file = uploader.value[0]
    
    # Convert memoryview to bytes
    content_mv = uploaded_file["content"]
    img_bytes = bytes(content_mv)
    
    # Base64 encode for the model
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    
    return img_b64, uploaded_file["type"]

# Process the image
img_b64, file_type = process_uploaded_image()

if img_b64:
    print("Image processed successfully and ready for analysis!")
else:
    print("Please upload an image first.")

Please upload an image first.


In [16]:
if img_b64:
    # Create a multimodal message with both text and image
    multimodal_message = HumanMessage(content=[
        {
            "type": "text", 
            "text": "Here's a photo of my pet! Please analyze their appearance, size, breed characteristics, and any personality traits you can observe, then recommend Christmas gifts that would be perfect for them. Include specific products and where to buy them."
        },
        {
            "type": "image", 
            "base64": img_b64, 
            "mime_type": file_type
        }
    ])
    
    # Send to our agent
    response = agent.invoke(
        {"messages": [multimodal_message]},
        config
    )
    
    print(response['messages'][-1].content)
else:
    print("Please upload a pet photo first!")

Please upload a pet photo first!


## Step 11: Monitoring with LangSmith (Optional)

LangSmith provides powerful tracing and debugging capabilities for your agents. If you have LangSmith set up, you can monitor:

- **Tool usage**: See which tools your agent calls and why
- **Performance metrics**: Track response times and token usage
- **Debugging**: Inspect the full reasoning chain when things go wrong
- **User feedback**: Collect ratings and improve your agent over time

To enable LangSmith tracing, uncomment the lines in Step 1 and set your LangSmith API key in your `.env` file:

```
LANGCHAIN_API_KEY=your_langsmith_api_key_here
```

Once enabled, you can view traces at [smith.langchain.com](https://smith.langchain.com)

## Key Learnings: Chatbots vs Agents

Through this tutorial, we've seen the evolution from simple chatbots to powerful AI agents:

### Chatbots (Step 2)
- ✅ Good for: Conversations, explanations, advice based on training data
- ❌ Limited by: Static knowledge, no real-time information, can't take actions

### AI Agents (Steps 4-11)
- ✅ Can: Search the web, find current prices, locate nearby stores, process images
- ✅ Provide: Real-time information, location-aware recommendations, actionable results
- ✅ Scale: Add new tools easily, monitor performance, stream responses

### When to Use Each
- **Use chatbots** for: FAQ systems, educational content, creative writing
- **Use agents** for: Shopping assistance, research tasks, data analysis, real-world actions

The key insight: **Agents = Chatbots + Tools + Real-world capabilities!**

## Step 12: Deploying to LangSmith

To deploy this agent to LangSmith for interactive use:

1. Make sure your `.env` file has all required API keys
2. Use the provided `langgraph_pet_gift.json` configuration file
3. Deploy with: `langgraph deploy --config langgraph_pet_gift.json`

Your agent will then be available through LangSmith's web interface for interactive chat!